In [ ]:
from sqlalchemy import (
    create_engine,
    Column,
    Integer,
    String,
    DateTime,
    Boolean,
    ForeignKey,
    Text,
)
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship
import pandas as pd
import os
from datetime import datetime
import logging

# Configure basic logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Load environment variables if needed
DATABASE_URL = os.getenv("DATABASE_URL", "sqlite:///app.db")

# Create SQLAlchemy base
Base = declarative_base()


# Define models
class User(Base):
    __tablename__ = "users"

    id = Column(Integer, primary_key=True)
    username = Column(String(50), unique=True, nullable=False)
    email = Column(String(100), unique=True, nullable=False)
    password_hash = Column(String(128), nullable=False)
    created_at = Column(DateTime, default=datetime.utcnow)
    is_active = Column(Boolean, default=True)

    # Relationships
    posts = relationship("Post", back_populates="author")

    def __repr__(self):
        return f"<User(username='{self.username}', email='{self.email}')>"


class Post(Base):
    __tablename__ = "posts"

    id = Column(Integer, primary_key=True)
    title = Column(String(100), nullable=False)
    content = Column(Text, nullable=False)
    user_id = Column(Integer, ForeignKey("users.id"), nullable=False)
    created_at = Column(DateTime, default=datetime.utcnow)
    updated_at = Column(DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)

    # Relationships
    author = relationship("User", back_populates="posts")

    def __repr__(self):
        return f"<Post(title='{self.title}', author_id={self.user_id})>"


# Database connection class
class Database:
    def __init__(self, db_url=None):
        self.engine = create_engine(db_url or DATABASE_URL)
        self.Session = sessionmaker(bind=self.engine)
        self.session = None

    def connect(self):
        """Establish connection to the database"""
        try:
            # Test connection
            self.engine.connect()
            logger.info("Successfully connected to database")
            self.session = self.Session()
            return True
        except Exception as e:
            logger.error(f"Database connection error: {e}")
            return False

    def create_tables(self):
        """Create all tables defined by the models"""
        try:
            Base.metadata.create_all(self.engine)
            logger.info("Database tables created successfully")
            return True
        except Exception as e:
            logger.error(f"Error creating tables: {e}")
            return False

    def close(self):
        """Close the database connection"""
        if self.session:
            self.session.close()
            logger.info("Database connection closed")

    def add_user(self, username, email, password_hash):
        """Add a new user to the database"""
        try:
            user = User(username=username, email=email, password_hash=password_hash)
            self.session.add(user)
            self.session.commit()
            logger.info(f"User {username} added successfully")
            return user
        except Exception as e:
            self.session.rollback()
            logger.error(f"Error adding user: {e}")
            return None

    def get_user_by_username(self, username):
        """Get user by username"""
        return self.session.query(User).filter(User.username == username).first()

    def get_user_by_email(self, email):
        """Get user by email"""
        return self.session.query(User).filter(User.email == email).first()

    def add_post(self, title, content, user_id):
        """Add a new post to the database"""
        try:
            post = Post(title=title, content=content, user_id=user_id)
            self.session.add(post)
            self.session.commit()
            logger.info(f"Post '{title}' added successfully")
            return post
        except Exception as e:
            self.session.rollback()
            logger.error(f"Error adding post: {e}")
            return None

    def get_posts_by_user(self, user_id):
        """Get all posts by a specific user"""
        return self.session.query(Post).filter(Post.user_id == user_id).all()

    def export_to_dataframe(self, table_name):
        """Export a table to pandas DataFrame"""
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql(query, self.engine)
            return df
        except Exception as e:
            logger.error(f"Error exporting to DataFrame: {e}")
            return None


# Singleton instance
_db_instance = None


def get_db():
    """Get or create a database instance"""
    global _db_instance
    if _db_instance is None:
        _db_instance = Database()
        _db_instance.connect()
    return _db_instance